# *Model de clasificare a imaginilor OCT folosind MobileNetV2*
***Este implementat în TensorFlow/Keras***

**1. Descărcare dataset**



In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/kermany2018")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'kermany2018' dataset.
Path to dataset files: /kaggle/input/kermany2018


**2. Mutare dataset**

In [6]:
import os
import shutil

BASE = "/content/oct"
RAW = os.path.join(BASE, "raw")

os.makedirs(BASE, exist_ok=True)

# Copiez datele
shutil.copytree(path, RAW, dirs_exist_ok=True)

print("RAW structure:")
for root, dirs, files in os.walk(RAW):
    print(root, dirs[:5])
    break


RAW structure:
/content/oct/raw ['oct2017', 'OCT2017 ']


**3. Preprocesare și pregătire date**


*a) Preprocesare*
* Transformare imagine în grayscale.
* Redimensionare imagine la (224, 224).
* Asigurare că pixelii sunt uint8.
* Aplicare denoise subtil cu medianBlur.
* Aplicare CLAHE pentru contrast adaptiv.
* Normalizare pixelii la [0, 1].
* Transformare grayscale-ul în 3 canale (pentru modele care așteaptă RGB).





In [13]:
import cv2
import numpy as np

TARGET_SIZE = (224, 224)

def preprocess_oct(image):
    # grayscale
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # resize
    image = cv2.resize(image, TARGET_SIZE)

    # uint8
    if image.dtype != np.uint8:
        image = (image * 255).astype(np.uint8) if image.max() <= 1.0 else image.astype(np.uint8)

    # denoise subtil
    image = cv2.medianBlur(image, 3)

    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    image = clahe.apply(image)

    # normalize 0-1
    image = image.astype("float32") / 255.0

    # convert to 3 channels
    return np.stack([image, image, image], axis=-1)


In [15]:
import tensorflow as tf

def preprocess_tf(image, label):
    image = tf.numpy_function(preprocess_oct, [image], Tout=tf.float32) #tensor în vector numPy
    image.set_shape([224, 224, 3])
    return image, label


*b) Pregătire date și aplicare preprocesare*

In [14]:
import tensorflow as tf

DATASET = "/content/oct/raw/OCT2017 /"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# train
train_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET + "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# val
val_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET + "val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# test
test_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET + "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

#preprocesare
def prepare_ds(ds):
    return ds.unbatch().map(preprocess_tf, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = prepare_ds(train_ds_raw)
val_ds   = prepare_ds(val_ds_raw)
test_ds  = prepare_ds(test_ds_raw)


Found 83484 files belonging to 4 classes.
Found 32 files belonging to 4 classes.
Found 968 files belonging to 4 classes.


**4. Creare și compilare model**

In [17]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

def make_cnn_model():
    base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(4, activation="softmax")
    ])
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

cnn_model = make_cnn_model()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


**5. Antrenare model**

In [18]:
history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)


Epoch 1/10
   2609/Unknown 343s 125ms/step - accuracy: 0.8113 - loss: 0.5224

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


2609/2609 ━━━━━━━━━━━━━━━━━━━━ 348s 127ms/step - accuracy: 0.8113 - loss: 0.5224 - val_accuracy: 0.9062 - val_loss: 0.2091
Epoch 2/10
2609/2609 ━━━━━━━━━━━━━━━━━━━━ 345s 118ms/step - accuracy: 0.8719 - loss: 0.3554 - val_accuracy: 0.9688 - val_loss: 0.1882
Epoch 3/10
2609/2609 ━━━━━━━━━━━━━━━━━━━━ 308s 118ms/step - accuracy: 0.8794 - loss: 0.3380 - val_accuracy: 0.9375 - val_loss: 0.1856
Epoch 4/10
2609/2609 ━━━━━━━━━━━━━━━━━━━━ 312s 119ms/step - accuracy: 0.8842 - loss: 0.3251 - val_accuracy: 0.9375 - val_loss: 0.1268
Epoch 5/10
2609/2609 ━━━━━━━━━━━━━━━━━━━━ 308s 118ms/step - accuracy: 0.8855 - loss: 0.3224 - val_accuracy: 0.9375 - val_loss: 0.1587
Epoch 6/10
2609/2609 ━━━━━━━━━━━━━━━━━━━━ 307s 118ms/step - accuracy: 0.8862 - loss: 0.3165 - val_accuracy: 0.9375 - val_loss: 0.1501
Epoch 7/10
2609/2609 ━━━━━━━━━━━━━━━━━━━━ 382s 146ms/step - accuracy: 0.8891 - loss: 0.3113 - val_accuracy: 0.9375 - val_loss: 0.1508
Epoch 8/10
2609/2609 ━━━━━━━━━━━━━━━━━━━━ 306s 117ms/step - accuracy: 0.8

**6. Testare model**

In [21]:
test_loss, test_acc = cnn_model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc:.4f}%")
print(f"Test Loss: {test_loss:.4f}")


31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 118ms/step - accuracy: 0.9482 - loss: 0.1337
Test Accuracy: 0.9267%
Test Loss: 0.1743


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


**7. Salvare model**

In [ ]:
cnn_model.save("oct_cnn_model_MobileNetV2.h5")
